In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import DateType, FloatType, IntegerType, TimestampType
from pyspark.sql.functions import col, trim, when


def normalize_weather(df: DataFrame) -> DataFrame:
    return ( df
            .filter((col('event_date') != '') & col('event_date').isNotNull())
            .withColumn("event_date", col('event_date').cast(DateType()))
            .withColumn("region", trim(col('region')))
            .withColumn("temperature_c", col('temperature_c').cast(FloatType()))
            .withColumn("wind_speed_kmh", col('wind_speed_kmh').cast(FloatType()))
            .withColumn("precipitation_mm", col('precipitation_mm').cast(FloatType()))
            .withColumn("last_updated_ts", col('last_updated_ts').cast(TimestampType()))
            .drop(col('_rescued_data'))
    )


def derived_temperature_column(df: DataFrame) -> DataFrame:
    return (df
            .withColumn('temp_category', 
                        when(col('temperature_c') < 0, "Frost")
                        .when(col('temperature_c') < 10, 'Cold')
                        .when(col('temperature_c') < 20, 'Mild')
                        .when(col('temperature_c') < 30, 'Warm')
                        .otherwise('Extreme Warm')
            )
    )

def derived_wind_strength_column(df: DataFrame) -> DataFrame:
    return (df
            .withColumn('wind_strength', 
                        when(col('wind_speed_kmh') < 0, "Invalid")
                        .when(col('wind_speed_kmh') < 30, 'Moderate')
                        .when(col('wind_speed_kmh') < 50, 'Strong')
                        .otherwise('Storm')
            )
    )


def derived_precipitation_column(df: DataFrame) -> DataFrame:
    return (df
            .withColumn('precip_intensity', 
                        when(col('precipitation_mm') < 0.5, "Light")
                        .when(col('precipitation_mm') < 4, 'Moderate')
                        .when(col('precipitation_mm') < 8, 'Heavy')
                        .otherwise('Violent')
            )
    )

def rename_columns(df: DataFrame) -> DataFrame:
        return (df
        .withColumnRenamed('region', 'weather_region')
        .withColumnRenamed('last_updated_ts', 'weather_last_updated_ts')
        .withColumnRenamed('source_system', 'weather_source_system')
        .withColumnRenamed('event_date', 'weather_event_date')
        )